# Phase 5 — Research Writeup Rigor

Two deliverables: (1) `docs/THREATS_TO_VALIDITY.md` — a written threats-to-validity section, referenced (not duplicated) here; (2) an empirical monolithic-vs-decomposed agent comparison.

**Important scope note on (2)**: with no `GROQ_API_KEY` configured, the "monolithic" side of this comparison falls back to a random mock weight (see `scaata/agents/monolithic.py::mock_monolithic_pass`) — the numbers below only prove the harness runs end-to-end, they are **not** a real finding about LLM self-assessment quality. Set `GROQ_API_KEY` in `.env` and re-run for the actual research result.

> ## ⚠ STATUS: partial mock — the monolithic comparison is not yet a real finding
>
> - **Threats-to-validity document**: real and complete, no caveats.
> - **Monolithic-vs-decomposed comparison**: the harness runs end-to-end and is unit-tested, but the "monolithic" LLM weight below is a **random mock value** (no live `GROQ_API_KEY` configured) — the correlation numbers prove the harness works, not that monolithic reasoning is actually worse.
>
> See the project [README](../README.md) for the full status table across all phases.

In [1]:
import sys
sys.path.insert(0, '..')
import os

from scaata.config import FEATURE_COLUMNS
from scaata.data.loaders import load_market_data
from scaata.features.technical import add_features
from scaata.features.normalize import normalize_data
from scaata.strategies.scraper import mock_strategies, scrape_strategies, filter_and_dedup
from scaata.evaluation.agent_comparison import compare_monolithic_vs_decomposed

HAS_LLM_ACCESS = bool(os.environ.get('GROQ_API_KEY'))
print(f"GROQ_API_KEY configured: {HAS_LLM_ACCESS} "
      f"{'(real comparison)' if HAS_LLM_ACCESS else '(mock fallback -- harness-only run)'}")

GROQ_API_KEY configured: False (mock fallback -- harness-only run)


## 1. Threats to validity

See [`docs/THREATS_TO_VALIDITY.md`](../docs/THREATS_TO_VALIDITY.md) for the full written section covering: survivorship bias, look-ahead bias in regime labeling, GDELT coverage drift, GDELT entity-matching precision, LLM non-determinism/version drift, Phase 4 small-sample risk, Phase 4 oracle-leakage risk, and reproducibility/data versioning — each with the specific code/test that mitigates it and the residual risk that remains.

## 2. Monolithic-vs-decomposed comparison

In [2]:
raw_df = load_market_data(['AAPL'], '2020-01-01', '2022-01-01', use_cache=True)
featured_df = add_features(raw_df)
train_df, test_df, mean, std = normalize_data(featured_df[:400], featured_df[400:], FEATURE_COLUMNS)

raw_strategies = mock_strategies()
result = compare_monolithic_vs_decomposed(train_df, raw_strategies, FEATURE_COLUMNS, max_iterations=4)

print(f"Decomposed pipeline: weight-vs-actual-profitability correlation = {result['decomposed_weight_vs_profit_corr']:.3f}")
print(f"Monolithic LLM pass: weight-vs-actual-profitability correlation = {result['monolithic_weight_vs_profit_corr']:.3f}")
print(f"\nTotal LLM calls -- decomposed: {result['n_decomposed_llm_calls']}, monolithic: {result['n_monolithic_llm_calls']}")
print("(Equal, because the decomposed pipeline's extra weight refinement comes from numeric "
      "meta-selector training + rule-based critique, not additional LLM calls -- if this holds "
      "up under a real LLM run too, it means decomposition buys more principled, "
      "performance-grounded weighting at no extra LLM cost.)")
print(f"\nDecomposed weights: {result['decomposed_weights']}")
print(f"Monolithic weights: {result['monolithic_weights']}")

if not HAS_LLM_ACCESS:
    print("\n*** Reminder: monolithic weights above are RANDOM MOCK values, not a real LLM self-assessment. ***")
    print("*** This run only demonstrates the harness works end-to-end -- re-run with GROQ_API_KEY for the real comparison. ***")

Decomposed pipeline: weight-vs-actual-profitability correlation = 0.923
Monolithic LLM pass: weight-vs-actual-profitability correlation = -0.707

Total LLM calls -- decomposed: 3, monolithic: 3
(Equal, because the decomposed pipeline's extra weight refinement comes from numeric meta-selector training + rule-based critique, not additional LLM calls -- if this holds up under a real LLM run too, it means decomposition buys more principled, performance-grounded weighting at no extra LLM cost.)

Decomposed weights: {'mock:ma_crossover': 0.5, 'mock:momentum': 1.0, 'mock:mean_reversion': 1.0}
Monolithic weights: {'mock:ma_crossover': 0.6547846749285817, 'mock:momentum': 0.5079146855055482, 'mock:mean_reversion': 0.4163894095744779}

*** Reminder: monolithic weights above are RANDOM MOCK values, not a real LLM self-assessment. ***
*** This run only demonstrates the harness works end-to-end -- re-run with GROQ_API_KEY for the real comparison. ***


## 3. Optional stretch (explicitly not implemented here)

A short live/paper-trading forward test — using a broker sandbox over a few real weeks going forward from today — would be the strongest available evidence of no look-ahead leakage, since the model has zero opportunity to have seen that data during any development step. This is left as a recommended next step rather than built here: it requires the user's own broker sandbox account and credentials, which per this project's safety practice should be set up and authorized by the user directly, not automated.

## Done criteria for this phase
- Threats-to-validity document exists, is specific (names files/tests), and is not vague boilerplate.
- Monolithic-vs-decomposed harness runs end-to-end for both schemes (verified above and by `tests/test_monolithic_comparison.py`).
- The mock-vs-live distinction for the comparison's actual research validity is stated explicitly, not buried.